<div dir="rtl">

# تطبيق K-Means لليوم الثاني — التعلم غير الخاضع للإشراف
كل خلية تجيب عن سؤال واحد. لا توجد أكواد ناقصة.

</div>

<div dir="rtl">

## قبل البدء: افتح جلسة نظيفة
شغّل الخلية التالية مرة واحدة لمسح أي متغيرات بقيت في ذاكرة Colab من تشغيل سابق. بعدها شغّل الخلايا بالترتيب من الأعلى إلى الأسفل.

</div>

In [ ]:
%reset -f
print("تم بدء جلسة نظيفة ✅")

<div dir="rtl">

## 0) تحميل البيانات

</div>

<div dir="rtl">

### 0.1 استيراد pandas

</div>

In [ ]:
import pandas as pd

<div dir="rtl">

### 0.2 قراءة الملف

</div>

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/hanenalmayouf/FAML/main/public/data/manafeth_customers.csv'
df = pd.read_csv(DATA_URL)
print('تم التحميل ✅', df.shape)

<div dir="rtl">

## 1) اختيار الخصائص

</div>

<div dir="rtl">

### 1.1 تحديد أسماء الخصائص السلوكية

</div>

In [ ]:
features = [
    'orders_per_month',
    'avg_basket_sar',
    'days_since_last_order',
    'promo_usage_rate'
]

<div dir="rtl">

### 1.2 إنشاء X من دون هدف y

</div>

In [ ]:
X_cluster = df[features].copy()

<div dir="rtl">

### 1.3 مشاهدة ملخص الخصائص

</div>

In [ ]:
display(X_cluster.describe().round(2))

<div dir="rtl">

## 2) تجهيز الأرقام

</div>

<div dir="rtl">

### 2.1 تعويض المفقود بالوسيط

</div>

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy='median')
X_filled = imputer.fit_transform(X_cluster)

<div dir="rtl">

### 2.2 تطبيق التحجيم (Scaling)

</div>

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_filled)

<div dir="rtl">

**لماذا؟** K-Means تعتمد على المسافة، لذلك نوحّد المقاييس قبل التجميع.

</div>

<div dir="rtl">

## 3) مقارنة قيم K

</div>

<div dir="rtl">

### 3.1 استيراد الأدوات

</div>

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

<div dir="rtl">

### 3.2 تجربة K من 2 إلى 6
هذه الخلية تكرر **الفكرة نفسها** لخمس قيم، لذلك وجود الحلقة هنا مقصود.

</div>

In [ ]:
k_results = []
for k in range(2, 7):
    candidate = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = candidate.fit_predict(X_scaled)
    k_results.append({
        'K': k,
        'Silhouette': silhouette_score(X_scaled, labels)
    })

<div dir="rtl">

### 3.3 عرض جدول المقارنة

</div>

In [ ]:
k_table = pd.DataFrame(k_results)
display(k_table.round(3))

<div dir="rtl">

## 4) بناء المثال النهائي باستخدام K = 3

</div>

<div dir="rtl">

### 4.1 إنشاء النموذج

</div>

In [ ]:
kmeans = KMeans(n_clusters=3, n_init=10, random_state=42)

<div dir="rtl">

### 4.2 اكتشاف المجموعات

</div>

In [ ]:
cluster_labels = kmeans.fit_predict(X_scaled)

<div dir="rtl">

### 4.3 إضافة رقم المجموعة

</div>

In [ ]:
clustered_df = df.copy()
clustered_df['cluster'] = cluster_labels

<div dir="rtl">

### 4.4 حساب متوسط كل خاصية داخل كل مجموعة

</div>

In [ ]:
cluster_profile = (
    clustered_df.groupby('cluster')[features]
    .mean()
    .round(2)
)
display(cluster_profile)

<div dir="rtl">

### 4.5 معرفة الخاصية الأكثر اختلافًا بين المجموعات

</div>

In [ ]:
feature_range = cluster_profile.max() - cluster_profile.min()
strongest_feature = feature_range.idxmax()
print('أوضح خاصية تفصل المجموعات:', strongest_feature)

<div dir="rtl">

## الخلاصة
اخترنا الخصائص، عوضنا المفقود، طبقنا Scaling، قارنا قيم K، ثم فسرنا المجموعات من متوسطاتها.

</div>